# rag inference
runs all 3 classifiers on llm justification texts, both standard and with rag
rag retrieves top 3 similar training speeches via faiss and prepends them as context

In [ ]:
! pip install transformers faiss-cpu sentence-transformers sentencepiece 

In [ ]:
import os
import ast
import numpy as np
import pandas as pd
import torch
import faiss
from transformers import(BertTokenizer, BertForSequenceClassification,AutoTokenizer, AutoModelForSequenceClassification, T5Tokenizer, T5ForConditionalGeneration)
from sentence_transformers import SentenceTransformer
import warnings; warnings.filterwarnings('ignore')

In [ ]:
device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# training data paths used to build the faiss retrieval index
us_train_path='/kaggle/input/datasets/zainabrizvi5/thesis-data/final_dataset/us_train.csv'
uk_train_path= '/kaggle/input/datasets/zainabrizvi5/thesis-data/final_dataset/uk_train.csv'
data_paths= {
    'gpt_us': '/kaggle/input/datasets/zainabrizvi5/llm-response/llm_responses/gpt_us.csv',
    'gpt_uk':'/kaggle/input/datasets/zainabrizvi5/llm-response/llm_responses/gpt_uk.csv',
    'claude_us':'/kaggle/input/datasets/zainabrizvi5/llm-response/llm_responses/claude_us.csv',
    'claude_uk': '/kaggle/input/datasets/zainabrizvi5/llm-response/llm_responses/claude_uk.csv',
    'llama_us':'/kaggle/input/datasets/zainabrizvi5/llm-response/llm_responses/llama_us.csv',
    'llama_uk':'/kaggle/input/datasets/zainabrizvi5/llm-response/llm_responses/llama_uk.csv',
}
model_paths= {
    'politbert_us':'/kaggle/input/thesis-models/politbert_us',
    'politbert_uk': '/kaggle/input/thesis-models/politbert_uk',
    'politics_us':'/kaggle/input/thesis-models/politics_us',
    'politics_uk': '/kaggle/input/thesis-models/politics_uk',
    't5_us': '/kaggle/input/thesis-models/t5_us',
    't5_uk':'/kaggle/input/thesis-models/t5_uk',
}
output_dir= '/kaggle/working/rag_results_v2'
os.makedirs(output_dir, exist_ok=True)

In [ ]:
top_k= 3
max_length= 256
embed_model= 'all-MiniLM-L6-v2'
us_id2label= {0: 'left', 1: 'right'}
uk_id2label= {0: 'left', 1: 'center', 2: 'right'}
us_valid_labels= ['left', 'right']
uk_valid_labels= ['left', 'center', 'right']

In [ ]:
embedder= SentenceTransformer(embed_model)
print('embedder loaded')

In [ ]:
# sample 20k speeches per country for the index - more than enough for retrieval
us_train_df= pd.read_csv(us_train_path).sample(n=20000, random_state=42).reset_index(drop=True)
uk_train_df= pd.read_csv(uk_train_path).sample(n=20000, random_state=42).reset_index(drop=True)
print(f'us: {len(us_train_df)}, uk: {len(uk_train_df)}')

In [ ]:
def build_index(texts, embedder):
    embeddings= embedder.encode(texts, batch_size=256, show_progress_bar=True,convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(embeddings)
    index= faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    print(f'index ready: {index.ntotal} vectors')
    return index
print('building us index')
us_index= build_index(us_train_df['text'].astype(str).tolist(), embedder)
print('building uk index')
uk_index= build_index(uk_train_df['text'].astype(str).tolist(), embedder)

In [ ]:
def retrieve(query, index, train_df, embedder, k=3):
    q= embedder.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(q)
    scores, indices= index.search(q, k)
    results= []
    for idx, score in zip(indices[0], scores[0]):
        if idx >= 0:
            results.append((str(train_df.iloc[idx]['text'])[:300], train_df.iloc[idx]['label'], float(score)))
    return results
def build_rag_input(query, retrieved):
    context= ' | '.join([f'[{label}] {text}' for text, label, _ in retrieved])
    return f'Context: {context} Query: {query}'

In [ ]:
# load all 6 trained models
politbert_us_tok=BertTokenizer.from_pretrained(model_paths['politbert_us'])
politbert_us_model= BertForSequenceClassification.from_pretrained(model_paths['politbert_us']).to(device).eval()
politbert_uk_tok= BertTokenizer.from_pretrained(model_paths['politbert_uk'])
politbert_uk_model= BertForSequenceClassification.from_pretrained(model_paths['politbert_uk']).to(device).eval()
politics_us_tok= AutoTokenizer.from_pretrained(model_paths['politics_us'])
politics_us_model= AutoModelForSequenceClassification.from_pretrained(model_paths['politics_us']).to(device).eval()
politics_uk_tok= AutoTokenizer.from_pretrained(model_paths['politics_uk'])
politics_uk_model= AutoModelForSequenceClassification.from_pretrained(model_paths['politics_uk']).to(device).eval()
t5_us_tok= T5Tokenizer.from_pretrained(model_paths['t5_us'])
t5_us_model= T5ForConditionalGeneration.from_pretrained(model_paths['t5_us']).to(device).eval()
t5_uk_tok=T5Tokenizer.from_pretrained(model_paths['t5_uk'])
t5_uk_model= T5ForConditionalGeneration.from_pretrained(model_paths['t5_uk']).to(device).eval()

In [ ]:
def classify(text, model, tokenizer, id2label, max_length=256):
    enc= tokenizer(text, max_length=max_length, padding='max_length', truncation=True, return_tensors='pt')
    with torch.no_grad():
        out= model(input_ids=enc['input_ids'].to(device), attention_mask=enc['attention_mask'].to(device))
        probs= torch.softmax(out.logits,dim=1)
        pred = torch.argmax(probs, dim=1).item()
    return id2label[pred], round(probs[0][pred].item(), 4)
def classify_t5(text, model, tokenizer, valid_labels, max_length=256):
    # t5 generates the label as text rather than using a classification head
    enc= tokenizer(f'classify political ideology: {text}', max_length=max_length, padding='max_length', truncation=True, return_tensors='pt')
    with torch.no_grad():
        gen= model.generate(input_ids=enc['input_ids'].to(device),attention_mask=enc['attention_mask'].to(device), max_new_tokens=5)
    pred= tokenizer.decode(gen[0], skip_special_tokens=True).strip().lower()
    if pred not in valid_labels:
        pred= valid_labels[0]
    return pred, 'N/A'

In [ ]:
# load the llm response csvs
llm_data= {}
for name,path in data_paths.items():
    df = pd.read_csv(path)
    df.columns= [c.strip().lower() for c in df.columns]
    df = df.dropna(subset=['justification']).reset_index(drop=True)
    df['justification']= df['justification'].astype(str)
    df['choice']= df['choice'].astype(str).str.strip()
    llm_data[name]= df
    print(f'{name}: {len(df)} responses')

In [ ]:
print('running us inference')
us_results=[]
for llm_name in ['gpt_us', 'claude_us', 'llama_us']:
    df= llm_data[llm_name]
    llm= llm_name.replace('_us', '')
    print(f'{llm.upper()}: {len(df)} responses')
    for i,row in df.iterrows():
        query= row['justification']
        retrieved= retrieve(query, us_index, us_train_df, embedder, top_k)
        rag_input=build_rag_input(query, retrieved)
        retrieved_labels= [r[1] for r in retrieved]
        pb_std,_= classify(query, politbert_us_model,politbert_us_tok, us_id2label)
        pb_rag, _= classify(rag_input, politbert_us_model, politbert_us_tok, us_id2label)
        pol_std,_= classify(query,politics_us_model,politics_us_tok,us_id2label)
        pol_rag, _= classify(rag_input,politics_us_model,politics_us_tok,us_id2label)
        t5_std,_=classify_t5(query, t5_us_model, t5_us_tok, us_valid_labels)
        t5_rag,_= classify_t5(rag_input,t5_us_model, t5_us_tok, us_valid_labels)
        us_results.append({
            'llm': llm,'country': 'US', 'question': row['question'],
            'stated_choice': row['choice'],'justification': query,
            'retrieved_labels': str(retrieved_labels),
            'politbert_std':pb_std,'politbert_rag':pb_rag,
            'politics_std':pol_std, 'politics_rag':pol_rag,
            't5_std': t5_std,'t5_rag':t5_rag,
            'politbert_changed': pb_std!= pb_rag,
            'politics_changed':pol_std != pol_rag,
            't5_changed':t5_std!= t5_rag,
        })
us_df= pd.DataFrame(us_results)
print(f'us done: {len(us_df)} rows')

In [ ]:
print('running uk inference')
uk_results= []
for llm_name in ['gpt_uk', 'claude_uk', 'llama_uk']:
    df= llm_data[llm_name]
    llm= llm_name.replace('_uk','')
    print(f' {llm.upper()}: {len(df)} responses')
    for i, row in df.iterrows():
        query= row['justification']
        retrieved= retrieve(query, uk_index, uk_train_df, embedder, top_k)
        rag_input= build_rag_input(query, retrieved)
        retrieved_labels= [r[1] for r in retrieved]
        pb_std, _= classify(query,politbert_uk_model, politbert_uk_tok, uk_id2label)
        pb_rag, _= classify(rag_input, politbert_uk_model, politbert_uk_tok, uk_id2label)
        pol_std, _= classify(query, politics_uk_model,politics_uk_tok,uk_id2label)
        pol_rag, _=classify(rag_input,politics_uk_model,politics_uk_tok, uk_id2label)
        t5_std, _= classify_t5(query, t5_uk_model,t5_uk_tok,uk_valid_labels)
        t5_rag, _=classify_t5(rag_input,t5_uk_model, t5_uk_tok,uk_valid_labels)
        uk_results.append({
            'llm': llm, 'country': 'UK','question': row['question'],
            'stated_choice': row['choice'], 'justification': query,
            'retrieved_labels': str(retrieved_labels),
            'politbert_std': pb_std,'politbert_rag': pb_rag,
            'politics_std':pol_std,'politics_rag':pol_rag,
            't5_std':t5_std,'t5_rag': t5_rag,
            'politbert_changed':pb_std != pb_rag,
            'politics_changed': pol_std != pol_rag,
            't5_changed':t5_std!= t5_rag,
        })
uk_df= pd.DataFrame(uk_results)
print(f'uk done: {len(uk_df)} rows')

In [ ]:
all_results= pd.concat([us_df, uk_df], ignore_index=True)
all_results.to_csv(os.path.join(output_dir,'rag_bias_results_v2.csv'), index=False)
print(f'saved {len(all_results)} rows')

In [ ]:
# how many labels changed with rag vs without
for country in ['US', 'UK']:
    print(f'{country}:')
    sub= all_results[all_results['country'] == country]
    for model in ['politbert', 'politics', 't5']:
        changed= sub[f'{model}_changed'].sum()
        print(f' {model}: {changed}/{len(sub)} changed ({changed/len(sub)*100:.0f}%)')
    print()

In [ ]:
# check what labels faiss actually retrieved
print('retrieved label distribution:')
for country in ['US','UK']:
    sub= all_results[all_results['country'] ==country]
    all_labels= []
    for v in sub['retrieved_labels']:
        all_labels.extend(ast.literal_eval(v))
    from collections import Counter
    counts= Counter(all_labels)
    total= sum(counts.values())
    print(f' {country}: {dict(counts)}')
    print(f'percentages: { {k: f"{v/total*100:.1f}%" for k,v in counts.most_common()} }')

In [ ]:
# for neutral-stated responses specifically, check if rag changes things
neutral= all_results[all_results['stated_choice'] =='neutral']
print(f'total neutral responses: {len(neutral)}')
for country in ['US', 'UK']:
    cn= neutral[neutral['country'] ==country]
    if len(cn)== 0: continue
    print(f'\n{country} ({len(cn)} neutral):')
    for llm in cn['llm'].unique():
        ln= cn[cn['llm'] == llm]
        print(f' {llm}:')
        for model in ['politbert','politics', 't5']:
            std= ln[f'{model}_std'].value_counts().to_dict()
            rag= ln[f'{model}_rag'].value_counts().to_dict()
            print(f'.{model}: std={std} | rag={rag}')